In [1]:
"""Upload a PDF through the same Preview pipeline the frontend uses.

Flow: login → POST /api/upload → poll GET /api/pdf/jobs/{job_id} → GET /api/pdf/jobs/{job_id}/result
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import requests

BASE = "http://localhost:8000"
PDF_PATH = Path("/Users/alokhegde/Dhara/media_to_upload1762321618.pdf")
EMAIL = "alokhegde2221@gmail.com"
PASSWORD = "12345678"

assert PDF_PATH.is_file(), f"Missing PDF: {PDF_PATH}"
print(f"PDF: {PDF_PATH.name} ({PDF_PATH.stat().st_size:,} bytes)")

# Login (or signup if the user does not exist yet)
r = requests.post(f"{BASE}/api/login", json={"email": EMAIL, "password": PASSWORD})
if r.status_code == 401:
    r = requests.post(
        f"{BASE}/api/signup",
        json={"email": EMAIL, "password": PASSWORD, "name": "Notebook"},
    )
r.raise_for_status()
auth = r.json()
TOKEN = auth["token"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print(f"Logged in as {auth['email']} (role={auth.get('role')})")

PDF: media_to_upload1762321618.pdf (3,479,370 bytes)
Logged in as alokhegde2221@gmail.com (role=data_steward)


In [2]:
# Same endpoint the console uses after file pick: POST /api/upload (type=pdf)
with PDF_PATH.open("rb") as f:
    up = requests.post(
        f"{BASE}/api/upload",
        headers=HEADERS,
        data={"type": "pdf"},
        files={"file": (PDF_PATH.name, f, "application/pdf")},
    )
print("POST /api/upload", up.status_code)
up.raise_for_status()
upload = up.json()
JOB_ID = upload["job_id"]
print(json.dumps(upload, indent=2))
print(f"\njob_id = {JOB_ID}")

POST /api/upload 200
{
  "job_id": "3a9bcb4d74a74291a1ee84b39b387673",
  "source_type": "pdf"
}

job_id = 3a9bcb4d74a74291a1ee84b39b387673


In [3]:
# Poll until extraction finishes (same as Processing → Preview unlock)
STATUS = None
for i in range(600):
    st = requests.get(f"{BASE}/api/pdf/jobs/{JOB_ID}", headers=HEADERS)
    st.raise_for_status()
    job = st.json()
    STATUS = job.get("status")
    pct = job.get("percent") or job.get("progress", {}).get("percent")
    msg = job.get("message") or (job.get("progress") or {}).get("message") or ""
    if i % 5 == 0 or STATUS not in ("queued", "running", "processing"):
        print(f"[{i:03d}] status={STATUS} percent={pct} {msg}")
    if STATUS in ("done", "completed_with_errors", "error", "failed"):
        break
    time.sleep(2)
else:
    raise TimeoutError(f"Job {JOB_ID} still {STATUS} after polling")

if STATUS not in ("done", "completed_with_errors"):
    raise RuntimeError(f"Job failed: {json.dumps(job, indent=2)[:2000]}")
print("\nJob finished:")
print(json.dumps({k: job.get(k) for k in ("job_id", "status", "filename", "percent", "message", "source_type")}, indent=2))

[000] status=done percent=100 Complete

Job finished:
{
  "job_id": "3a9bcb4d74a74291a1ee84b39b387673",
  "status": "done",
  "filename": "media_to_upload1762321618.pdf",
  "percent": 100,
  "message": "Complete",
  "source_type": "pdf"
}


In [4]:
# Preview stage payload — same as PdfReview: GET /api/pdf/jobs/{job_id}/result
res = requests.get(f"{BASE}/api/pdf/jobs/{JOB_ID}/result", headers=HEADERS)
print("GET /api/pdf/jobs/.../result", res.status_code)
res.raise_for_status()
preview = res.json()
TABLES = preview.get("tables") or []

print(f"filename: {preview.get('filename')}")
print(f"is_batch: {preview.get('is_batch')}")
print(f"table count: {len(TABLES)}")
print()

# Compact card list (what the Preview UI shows in the left/list view)
rows = []
for t in TABLES:
    rows.append({
        "table_id": t.get("table_id"),
        "page": t.get("page"),
        "title": (t.get("title") or "")[:80],
        "n_cols": len(t.get("columns") or []),
        "n_rows": len(t.get("rows") or []),
        "human_review_needed": bool(t.get("human_review_needed")),
        "human_review_reason": t.get("human_review_reason"),
        "source_location": t.get("source_location"),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    print(json.dumps(rows, indent=2)[:8000])

GET /api/pdf/jobs/.../result 200
filename: media_to_upload1762321618.pdf
is_batch: False
table count: 162



,table_id,page,title,n_cols,n_rows,human_review_needed,human_review_reason,source_location
0,DDI_DES_DEL_VS_03447,3,Team Associated in the Preparation of Publication,2,5,False,None,Page 3
1,DDI_DES_DEL_VS_03448,7,"Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengal...",4,3,False,None,Page 7
2,DDI_DES_DEL_VS_03449,8,"Classification of Commercial, Fibre, and Horti...",4,3,True,title_filled_by_ai,Page 8
3,DDI_DES_DEL_VS_03450,9,"Area : In lakh hectares, Production: In lakh T...",8,19,False,None,Page 9
4,DDI_DES_DEL_VS_03451,10,"Area : In lakh hectares, Production: In lakh T...",8,25,False,None,Page 10
...,...,...,...,...,...,...,...,...
157,DDI_DES_DEL_VS_03604,163,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",8,32,False,None,Page 163
158,DDI_DES_DEL_VS_03605,164,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",17,32,False,None,Page 164
159,DDI_DES_DEL_VS_03606,165,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",11,32,False,None,Page 165
160,DDI_DES_DEL_VS_03607,166,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",8,32,False,None,Page 166


In [5]:
# Inspect one Preview table in full (columns + first rows) — pick by index
IDX = 0
t = TABLES[IDX]
print(f"table_id: {t.get('table_id')}")
print(f"title:    {t.get('title')}")
print(f"page:     {t.get('page')}")
print(f"review:   needed={t.get('human_review_needed')} reason={t.get('human_review_reason')}")
print()
cols = t.get("columns") or []
if cols and isinstance(cols[0], dict):
    col_names = [c.get("name") for c in cols]
else:
    col_names = list(cols)
print("columns:", col_names)
print(f"rows ({len(t.get('rows') or [])} total), first 8:")
for row in (t.get("rows") or [])[:8]:
    print(row)

# Full JSON for this table (can be large)
# print(json.dumps(t, indent=2, default=str)[:5000])

table_id: DDI_DES_DEL_VS_03447
title:    Team Associated in the Preparation of Publication
page:     3
review:   needed=False reason=None

columns: ['Sri K. Gururaja Rao', 'Additional Director']
rows (5 total), first 8:
['Sri. Shashidhara.T', 'Joint Director']
['Smt. Girija.T.S', 'Deputy Director']
['Smt. Ashwini kumari .H.B', 'Assistant Director']
['Smt. Mamatha .T.K', 'Assistant Statistical Officer']
['Kum.Nethravathi.M', 'Enumerator']


In [7]:
# Optional: skip re-upload — load Preview for an already-finished job_id
# (this PDF already ran as 7e77554c3af64c71809a0363e151304e → 162 tables)
JOB_ID = "3a9bcb4d74a74291a1ee84b39b387673"

res = requests.get(f"{BASE}/api/pdf/jobs/{JOB_ID}/result", headers=HEADERS)
res.raise_for_status()
preview = res.json()
TABLES = preview.get("tables") or []
print(f"job_id={JOB_ID}  filename={preview.get('filename')}  tables={len(TABLES)}")

# How many need human review (Preview badges)
n_review = sum(1 for t in TABLES if t.get("human_review_needed"))
print(f"human_review_needed: {n_review} / {len(TABLES)}")

# Persist a slim summary next to the notebook for inspection
summary_path = Path("/Users/alokhegde/Dhara/chat_preview_summary.json")
if summary_path.is_file():
    print(f"\nAlso on disk: {summary_path} ({summary_path.stat().st_size:,} bytes)")
    slim = json.loads(summary_path.read_text())
    print(f"cached summary table_count={slim.get('table_count')}")

job_id=3a9bcb4d74a74291a1ee84b39b387673  filename=media_to_upload1762321618.pdf  tables=162
human_review_needed: 1 / 162

Also on disk: /Users/alokhegde/Dhara/chat_preview_summary.json (186,948 bytes)
cached summary table_count=162


# Chat interface — llm_preview catalog

Send a **compact** table stub (no rows/columns) to `POST /api/llm_preview/jobs/{job_id}/llm_preview_data`, then read it back with GET.

Default POST stores `{table_id, name, summary}` from the stubs — **no LangChain**. Pass `"force_summarize": true` only if you want an LLM rewrite (needs API key).

In [14]:
# Compact stubs only — LLM does not need full Preview rows/columns
def compact_tables_for_llm(tables):
    out = []
    for t in tables:
        if not t.get("table_id"):
            continue
        entry = {
            "table_id": t.get("table_id"),
            "page": t.get("page"),
            "title": t.get("title"),
            "banner_table_id": t.get("banner_table_id"),
            "short_description": t.get("short_description") or t.get("description") or t.get("summary"),
            "human_review_needed": t.get("human_review_needed"),
            "human_review_reason": t.get("human_review_reason"),
        }
        # drop empty/None values to keep the payload small
        out.append({k: v for k, v in entry.items() if v is not None and v != ""})
    return out

CHAT_TABLES = compact_tables_for_llm(TABLES)
payload_bytes = len(json.dumps(CHAT_TABLES).encode("utf-8"))
full_bytes = len(json.dumps(TABLES, default=str).encode("utf-8"))
print(f"tables: {len(CHAT_TABLES)}")
print(f"compact payload: {payload_bytes:,} bytes  (full Preview tables would be ~{full_bytes:,} bytes)")
print("sample stub:")
print(json.dumps(CHAT_TABLES[0] if CHAT_TABLES else {}, indent=2))

# POST → stores catalog from stubs (no LangChain). Use force_summarize=True only if you want LLM rewrite.
post = requests.post(
    f"{BASE}/api/llm_preview/jobs/{JOB_ID}/llm_preview_data",
    headers={**HEADERS, "Content-Type": "application/json"},
    json={"tables": CHAT_TABLES},
    timeout=120,
)
print("\nPOST /api/llm_preview/jobs/.../llm_preview_data", post.status_code)
if not post.ok:
    print(post.text[:2000])
post.raise_for_status()
post_body = post.json()
print(f"returned catalog entries: {len(post_body.get('tables') or [])}")
print(f"source_table_count: {post_body.get('source_table_count')}")
print(f"updated_at: {post_body.get('updated_at')}")
print("\nfirst 5 catalog rows from POST response:")
print(json.dumps((post_body.get("tables") or [])[:5], indent=2))

tables: 162
compact payload: 30,941 bytes  (full Preview tables would be ~1,089,441 bytes)
sample stub:
{
  "table_id": "DDI_DES_DEL_VS_03447",
  "page": 3,
  "title": "Team Associated in the Preparation of Publication",
  "human_review_needed": false
}

POST /api/llm_preview/jobs/.../llm_preview_data 200
returned catalog entries: 162
source_table_count: 162
updated_at: 1792399816.2030818

first 5 catalog rows from POST response:
[
  {
    "table_id": "DDI_DES_DEL_VS_03447",
    "name": "Team Associated in the Preparation of Publication",
    "summary": "Team Associated in the Preparation of Publication"
  },
  {
    "table_id": "DDI_DES_DEL_VS_03448",
    "name": "Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengalgram, Groundnut, Sunflower, Safflower and Cotton crops are given for irrigated",
    "summary": "Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengalgram, Groundnut, Sunflower, Safflower and Cotton crops are given for irrigated"
  },
  {
    "table_id": "DDI_DES_DEL_VS_03449",
    "nam

In [15]:
# GET stored catalog (same job_id — what chat mode reads back)
get = requests.get(
    f"{BASE}/api/llm_preview/jobs/{JOB_ID}/llm_preview_data",
    headers=HEADERS,
    timeout=60,
)
print("GET /api/llm_preview/jobs/.../llm_preview_data", get.status_code)
if not get.ok:
    print(get.text[:2000])
get.raise_for_status()
llm_preview = get.json()
print("llm_preview",llm_preview)
CATALOG = llm_preview.get("tables") or []

print(f"job_id: {llm_preview.get('job_id')}")
print(f"catalog tables: {len(CATALOG)}")
print(f"pipeline_step: {llm_preview.get('pipeline_step')}")
print(f"last_turns: {len(llm_preview.get('last_turns') or [])}")
print()

try:
    import pandas as pd
    display(pd.DataFrame(CATALOG))
except Exception:
    print(json.dumps(CATALOG[:20], indent=2))

# Persist for inspection next to the notebook
out_path = Path("/Users/alokhegde/Dhara/chat_llm_preview_data.json")
out_path.write_text(json.dumps(llm_preview, indent=2, default=str))
print(f"\nwrote {out_path}")

GET /api/llm_preview/jobs/.../llm_preview_data 200
llm_preview {'job_id': '3a9bcb4d74a74291a1ee84b39b387673', 'tables': [{'table_id': 'DDI_DES_DEL_VS_03447', 'name': 'Team Associated in the Preparation of Publication', 'summary': 'Team Associated in the Preparation of Publication'}, {'table_id': 'DDI_DES_DEL_VS_03448', 'name': 'Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengalgram, Groundnut, Sunflower, Safflower and Cotton crops are given for irrigated', 'summary': 'Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengalgram, Groundnut, Sunflower, Safflower and Cotton crops are given for irrigated'}, {'table_id': 'DDI_DES_DEL_VS_03449', 'name': 'Classification of Commercial, Fibre, and Horticultural Crops', 'summary': 'Classification of Commercial, Fibre, and Horticultural Crops'}, {'table_id': 'DDI_DES_DEL_VS_03450', 'name': 'Area : In lakh hectares, Production: In lakh Tonnes, Yield : In Kgs per hectare', 'summary': 'Area : In lakh hectares, Production: In lakh Tonnes, Yield : In Kgs per hectar

,table_id,name,summary
0,DDI_DES_DEL_VS_03447,Team Associated in the Preparation of Publication,Team Associated in the Preparation of Publication
1,DDI_DES_DEL_VS_03448,"Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengal...","Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengal..."
2,DDI_DES_DEL_VS_03449,"Classification of Commercial, Fibre, and Horti...","Classification of Commercial, Fibre, and Horti..."
3,DDI_DES_DEL_VS_03450,"Area : In lakh hectares, Production: In lakh T...","Area : In lakh hectares, Production: In lakh T..."
4,DDI_DES_DEL_VS_03451,"Area : In lakh hectares, Production: In lakh T...","Area : In lakh hectares, Production: In lakh T..."
...,...,...,...
157,DDI_DES_DEL_VS_03604,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...","FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN..."
158,DDI_DES_DEL_VS_03605,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...","FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN..."
159,DDI_DES_DEL_VS_03606,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...","FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN..."
160,DDI_DES_DEL_VS_03607,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...","FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN..."



wrote /Users/alokhegde/Dhara/chat_llm_preview_data.json


# LangChain intent over llm_preview catalog

Load the stored catalog → LangChain structured output → `{ type, table_names }`.
Example prompt: *show me tables related to condiment and spices*.

In [ ]:
from typing import Any, Dict, List, Literal, Optional

# Ensure LangChain is available in this Jupyter kernel
try:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_openai import ChatOpenAI
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "langchain-core>=0.3,<1", "langchain-openai>=0.3,<1"]
    )
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_openai import ChatOpenAI

from pydantic import BaseModel, Field, field_validator

ACTION_TYPES = (
    "find",
    "bulk_find",
    "delete",
    "bulk_delete",
    "download",
    "bulk_download",
    "find_review_needed",
    "find_ai_review_needed",
)

SYSTEM_PROMPT = """You are a precise table-selection agent for a government statistics Preview catalog.

You receive:
1) A steward's natural-language request.
2) A catalog of extracted tables. Each entry has:
   - table_id: stable id (ALWAYS use this in table_names — never invent ids)
   - name: display title
   - summary: short description (often repeats the title)

Your job: choose an action type and the matching table_id values.

Action types:
- find — locate / show a small number of matching tables
- bulk_find — many matches (prefer when the request clearly covers a whole topic with several tables)
- delete / bulk_delete — identify tables to remove (selection only; do not invent deletes)
- download / bulk_download — prepare matching tables for download
- find_review_needed / find_ai_review_needed — only when the steward asks about review / AI flags

Matching rules (accuracy over recall):
- Match on name and summary: topic words, crop names, synonyms, and clear phrases
  (e.g. "condiment and spices" ↔ "CONDIMENT AND SPICE", "spice crops").
- Include a table ONLY if it clearly belongs to the request. Prefer precision.
- If several consecutive tables share the same topic title (multi-page series),
  include ALL of them that match — use bulk_find when there are 3+.
- Never invent table_ids. Every id MUST appear in the catalog.
- If nothing matches, return type=find and table_names=[].
- When unsure between find and delete, choose find.

Output ONLY the structured fields: type, table_names, rationale."""


class CatalogAction(BaseModel):
    type: Literal[
        "find",
        "bulk_find",
        "delete",
        "bulk_delete",
        "download",
        "bulk_download",
        "find_review_needed",
        "find_ai_review_needed",
    ] = Field(description="Action to perform over the catalog")
    table_names: List[str] = Field(
        default_factory=list,
        description="Catalog table_id values that match the request",
    )
    rationale: str = Field(default="", description="One short sentence explaining the selection")

    @field_validator("table_names", mode="before")
    @classmethod
    def _coerce_ids(cls, v):
        if v is None:
            return []
        if isinstance(v, str):
            return [v.strip()] if v.strip() else []
        if isinstance(v, list):
            return [str(x).strip() for x in v if x is not None and str(x).strip()]
        return []


def fetch_llm_preview_catalog(job_id: str, headers: dict, base: str = BASE) -> List[Dict[str, Any]]:
    """GET stored llm_preview catalog for this job."""
    r = requests.get(f"{base}/api/llm_preview/jobs/{job_id}/llm_preview_data", headers=headers, timeout=60)
    r.raise_for_status()
    body = r.json()
    tables = body.get("tables") or []
    if not tables:
        raise ValueError("llm_preview catalog is empty — POST llm_preview_data first")
    return tables


def resolve_openai_api_key(headers: dict, base: str = BASE) -> str:
    """Use Settings key from the backend (same key the app uses)."""
    r = requests.get(f"{base}/api/llm-key", headers=headers, timeout=30)
    r.raise_for_status()
    key = (r.json().get("api_key") or "").strip()
    if not key:
        raise ValueError("No LLM API key in Settings — save one before querying the catalog")
    return key


def query_preview_catalog(
    user_request: str,
    *,
    catalog: Optional[List[Dict[str, Any]]] = None,
    job_id: Optional[str] = None,
    api_key: Optional[str] = None,
    headers: Optional[dict] = None,
    base: str = BASE,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    """
    Run LangChain over the llm_preview catalog.

    Returns::

        {
          "type": "find" | "bulk_find" | ...,
          "table_names": { "<table_id>": "<name>", ... },  # id → display name
          "table_ids": ["..."],                             # ordered list
          "rationale": "...",
        }
    """
    text = (user_request or "").strip()
    if not text:
        raise ValueError("user_request is required")

    hdrs = headers or HEADERS
    if catalog is None:
        jid = job_id or JOB_ID
        catalog = fetch_llm_preview_catalog(jid, hdrs, base=base)

    by_id = {
        str(t["table_id"]): t
        for t in catalog
        if isinstance(t, dict) and t.get("table_id")
    }
    if not by_id:
        return {"type": "find", "table_names": {}, "table_ids": [], "rationale": "Empty catalog."}

    key = (api_key or "").strip() or resolve_openai_api_key(hdrs, base=base)

    # Compact JSON for the model — ids + names only (already the catalog shape).
    catalog_json = json.dumps(
        [{"table_id": tid, "name": (t.get("name") or ""), "summary": (t.get("summary") or "")[:240]}
         for tid, t in by_id.items()],
        ensure_ascii=False,
    )

    llm = ChatOpenAI(model=model, temperature=0, api_key=key)
    structured = llm.with_structured_output(CatalogAction)
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", SYSTEM_PROMPT),
            (
                "human",
                "Steward request:\n{user_request}\n\nCatalog JSON (use ONLY these table_id values):\n{catalog_json}",
            ),
        ]
    )
    plan: CatalogAction = (prompt | structured).invoke(
        {"user_request": text, "catalog_json": catalog_json}
    )

    # Sanitize: only ids that exist in the catalog
    ids: List[str] = []
    seen = set()
    for tid in plan.table_names:
        if tid in by_id and tid not in seen:
            seen.add(tid)
            ids.append(tid)

    names_dict = {tid: str(by_id[tid].get("name") or tid) for tid in ids}
    print("names_dict",names_dict)
    action = plan.type
    print("action",action)
    if action == "find" and len(ids) >= 3:
        action = "bulk_find"

    return {
        "type": action,
        "table_names": names_dict,
        "table_ids": ids,
        "rationale": (plan.rationale or "").strip(),
    }


print("Defined query_preview_catalog()")

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Defined query_preview_catalog()


In [23]:
# Example: find condiment / spice tables from the stored catalog
USER_PROMPT = "download tables related to condiment and spices"

result = query_preview_catalog(
    USER_PROMPT,
    catalog=CATALOG,  # from previous GET cell; or omit to re-fetch
    job_id=JOB_ID,
)

print("type:", result["type"])
print("rationale:", result["rationale"])
print(f"matches: {len(result['table_ids'])}")
print()
print("table_names (id → name):")
print(json.dumps(result["table_names"], indent=2, ensure_ascii=False))

type: bulk_download
rationale: These tables specifically relate to condiment and spice crops.
matches: 3

table_names (id → name):
{
  "DDI_DES_DEL_VS_03605": "FULLY REVISED ESTIMATES OF AREA, PRODUCTION AND YIELD OF CONDIMENT AND SPICE CROPS IN KARNATAKA FOR THE YEAR 2023-24",
  "DDI_DES_DEL_VS_03606": "FULLY REVISED ESTIMATES OF AREA, PRODUCTION AND YIELD OF CONDIMENT AND SPICECROPS IN KARNATAKA FOR THE YEAR 2023-24",
  "DDI_DES_DEL_VS_03607": "FULLY REVISED ESTIMATES OF AREA, PRODUCTION AND YIELD OF CONDIMENT AND SPICECROPS IN KARNATAKA"
}


/opt/homebrew/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=CatalogAction(type='bulk_...iment and spice crops.'), input_type=CatalogAction])
  return self.__pydantic_serializer__.to_python(


# Execute action from LangChain result

`query_preview_catalog` only returns `{ type, table_names, … }`.  
`execute_preview_action` runs the matching Preview operation on those ids.

In [24]:
from pathlib import Path
from typing import Any, Dict, List, Optional


def _live_preview_tables(
    *,
    tables: Optional[List[dict]] = None,
    job_id: Optional[str] = None,
    headers: Optional[dict] = None,
    base: str = BASE,
) -> List[dict]:
    """Prefer in-memory TABLES from Preview GET /result; else re-fetch."""
    if tables is not None:
        return tables
    if "TABLES" in globals() and TABLES:
        return TABLES
    jid = job_id or JOB_ID
    hdrs = headers or HEADERS
    r = requests.get(f"{base}/api/pdf/jobs/{jid}/result", headers=hdrs, timeout=120)
    r.raise_for_status()
    return r.json().get("tables") or []


def _subset_by_ids(tables: List[dict], table_ids: List[str]) -> Dict[str, dict]:
    want = {str(t) for t in table_ids}
    out = {}
    for t in tables:
        tid = str(t.get("table_id") or "")
        if tid in want:
            out[tid] = t
    return out


def _action_find(action, table_ids, live, **_):
    table_data = _subset_by_ids(live, table_ids)
    return {
        "type": action,
        "executed": True,
        "table_ids": list(table_data.keys()),
        "table_data": table_data,
        "count": len(table_data),
    }


def _action_delete(action, table_ids, live, *, confirm_delete, jid, hdrs, base, **_):
    table_data = _subset_by_ids(live, table_ids)
    if not confirm_delete:
        return {
            "type": action,
            "executed": False,
            "table_ids": list(table_data.keys()),
            "table_data": table_data,
            "count": len(table_data),
            "message": "Pass confirm_delete=True to soft-delete these tables",
        }
    if not table_ids:
        return {"type": action, "executed": False, "deleted": [], "count": 0}
    r = requests.post(
        f"{base}/api/pdf/jobs/{jid}/tables/delete",
        headers={**hdrs, "Content-Type": "application/json"},
        json={"table_ids": table_ids},
        timeout=60,
    )
    r.raise_for_status()
    body = r.json()
    return {
        "type": action,
        "executed": True,
        "deleted": body.get("deleted") or table_ids,
        "deleted_count": body.get("deleted_count"),
        "response": body,
    }


def _action_download(action, table_ids, live, *, jid, hdrs, base, download_dir, **_):
    if not table_ids:
        return {"type": action, "executed": False, "path": None, "count": 0}
    r = requests.post(
        f"{base}/api/pdf/jobs/{jid}/tables/download-zip",
        headers={**hdrs, "Content-Type": "application/json"},
        json={"table_ids": table_ids},
        timeout=180,
    )
    r.raise_for_status()
    out_dir = Path(download_dir) if download_dir else Path("/Users/alokhegde/Dhara")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{jid}_selected_tables.zip"
    out_path.write_bytes(r.content)
    return {
        "type": action,
        "executed": True,
        "path": str(out_path),
        "table_ids": table_ids,
        "count": len(table_ids),
        "bytes": len(r.content),
    }


def _action_find_review_needed(action, table_ids, live, **_):
    scope = _subset_by_ids(live, table_ids) if table_ids else {
        str(t.get("table_id")): t for t in live if t.get("table_id")
    }
    table_data = {tid: t for tid, t in scope.items() if t.get("human_review_needed")}
    return {
        "type": action,
        "executed": True,
        "table_ids": list(table_data.keys()),
        "table_data": table_data,
        "count": len(table_data),
    }


def _action_find_ai_review_needed(action, table_ids, live, **_):
    scope = _subset_by_ids(live, table_ids) if table_ids else {
        str(t.get("table_id")): t for t in live if t.get("table_id")
    }

    def _ai(t: dict) -> bool:
        if t.get("title_repaired_by_llm") or t.get("table_id_repaired_by_llm"):
            return True
        return str(t.get("human_review_reason") or "") in {
            "title_filled_by_ai",
            "uncertain_title",
            "id_title_mismatch",
        }

    table_data = {tid: t for tid, t in scope.items() if _ai(t)}
    return {
        "type": action,
        "executed": True,
        "table_ids": list(table_data.keys()),
        "table_data": table_data,
        "count": len(table_data),
    }


def execute_preview_action(
    action_result: Dict[str, Any],
    *,
    tables: Optional[List[dict]] = None,
    job_id: Optional[str] = None,
    headers: Optional[dict] = None,
    base: str = BASE,
    confirm_delete: bool = False,
    download_dir: Optional[Path] = None,
) -> Dict[str, Any]:
    """
    Run the Preview operation implied by ``query_preview_catalog`` output.

    - find / bulk_find → return full table objects for those ids (from Preview result)
    - delete / bulk_delete → soft-delete via API only if ``confirm_delete=True``
    - download / bulk_download → ZIP selected tables to disk
    - find_review_needed / find_ai_review_needed → filter by review flags
    """
    action = str(action_result.get("type") or "find").strip().lower()
    table_ids = list(action_result.get("table_ids") or [])
    if not table_ids and isinstance(action_result.get("table_names"), dict):
        table_ids = list(action_result["table_names"].keys())

    ctx = {
        "action": action,
        "table_ids": table_ids,
        "live": _live_preview_tables(
            tables=tables, job_id=job_id or JOB_ID, headers=headers or HEADERS, base=base
        ),
        "confirm_delete": confirm_delete,
        "jid": job_id or JOB_ID,
        "hdrs": headers or HEADERS,
        "base": base,
        "download_dir": download_dir,
    }

    match action:
        case "find" | "bulk_find":
            return _action_find(**ctx)
        case "delete" | "bulk_delete":
            return _action_delete(**ctx)
        case "download" | "bulk_download":
            return _action_download(**ctx)
        case "find_review_needed":
            return _action_find_review_needed(**ctx)
        case "find_ai_review_needed":
            return _action_find_ai_review_needed(**ctx)
        case _:
            raise ValueError(f"Unknown action type: {action!r}")


print("Defined execute_preview_action()")

Defined execute_preview_action()


In [25]:
# Run the action returned by LangChain (for bulk_find → load full Preview tables)
executed = execute_preview_action(result, tables=TABLES)

print("type:", executed["type"])
print("executed:", executed.get("executed"))
print("count:", executed.get("count"))
print("table_ids:", executed.get("table_ids"))

# Peek at first matched table (title + shape only — rows can be large)
if executed.get("table_data"):
    tid0 = executed["table_ids"][0]
    t0 = executed["table_data"][tid0]
    print()
    print(f"sample {tid0}:")
    print("  title:", t0.get("title"))
    print("  page:", t0.get("page"))
    print("  n_cols:", len(t0.get("columns") or []))
    print("  n_rows:", len(t0.get("rows") or []))

# Delete example (safe by default — will NOT delete unless you uncomment):
# execute_preview_action(result, tables=TABLES, confirm_delete=True)

# Download example:
# execute_preview_action({**result, "type": "bulk_download"}, tables=TABLES)

type: bulk_download
executed: True
count: 3
table_ids: ['DDI_DES_DEL_VS_03605', 'DDI_DES_DEL_VS_03606', 'DDI_DES_DEL_VS_03607']
